In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 
import statsmodels.formula.api as smf

# Read in synthetic data

df = pd.read_csv("data/raw/project_google_synth.csv")
user_level = pd.read_csv("data/raw/user_level.csv")

print(user_level.columns)

In [7]:
# identify promo period rows
promo = df[df["post"]==1].copy() 

# list of users that were exposed at least once 
treated_users = promo[promo["exposed"]==1]['user_id'].unique() 

# create the treatment indicator at the user level
df['ever_exposed'] = df['user_id'].isin(treated_users).astype(int)



### Step 1 — Merge Weights Back to Full Dataset

You computed weights at the user level.

Now attach them to each row of the panel.

In [8]:
# merge IPW weights into panel data
df_w = df.merge(
    user_level[['user_id','ipw']],
    on = "user_id",
    how = 'left'
)

df_w['ipw'].describe()

count    350000.000000
mean          0.999893
std           0.004410
min           0.996700
25%           0.997232
50%           0.998285
75%           1.000640
max           1.021756
Name: ipw, dtype: float64

### Step 2 - Create Interaction Term  

This is the DiD treatment variable

In [9]:
df_w['did'] = df_w['post'] * df_w['ever_exposed']

### Step 3 — Run Weighted DiD (WLS)
  
We use Weighted Least Squares

In [10]:
import statsmodels.api as sm  

# define regressors
X = df_w[['post', 'ever_exposed', 'did']]
X = sm.add_constant(X)

y = df_w['margin_dollars']
w = df_w['ipw']

# weighted regression
model_w = sm.WLS(y, X, weights=w).fit(
    cov_type='cluster',
    cov_kwds={'groups':df_w['user_id']}
)

print(model_w.summary())

                            WLS Regression Results                            
Dep. Variable:         margin_dollars   R-squared:                       0.022
Model:                            WLS   Adj. R-squared:                  0.022
Method:                 Least Squares   F-statistic:                     3122.
Date:                Mon, 02 Mar 2026   Prob (F-statistic):               0.00
Time:                        16:48:04   Log-Likelihood:            -1.2383e+06
No. Observations:              350000   AIC:                         2.477e+06
Df Residuals:                  349996   BIC:                         2.477e+06
Df Model:                           3                                         
Covariance Type:              cluster                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            3.3131      0.480      6.896   

### What This Model Is Doing (Plain English)

It’s:  

	    •	Comparing treated vs control trends (DiD),  
	    •	But giving different importance to different users (IPW),  
	    •	While adjusting uncertainty for repeated users (clustered SE).  

Same logic as before.  
Just reweighted.

## 6. Weighted Difference-in-Differences Results

### Model Comparison

| Model | Estimate | Standard Error |
|--------|----------|----------------|
| Unweighted DiD | -4.596 | 0.701 |
| Two-Way Fixed Effects (TWFE) | -4.596 | 0.646 |
| Weighted DiD (IPW) | -4.594 | 0.643 |
| True ATT (Synthetic Ground Truth) | -6.538 | — |

---

### Interpretation

The weighted DiD estimate (-4.594) is nearly identical to the unweighted DiD estimate (-4.596).

This indicates that inverse probability weighting (IPW) did not materially change the estimated treatment effect.

---

### Why Weighting Did Not Change the Result

Earlier diagnostics showed:

- Propensity scores were clustered near 1.0.
- There was limited overlap between treated and control users.
- Covariate balance did not improve meaningfully after weighting.

Because the campaign was highly targeted, the data lacks sufficient common support between treated and untreated customers. As a result:

- Reweighting does not create a substantially different pseudo-population.
- The weighted and unweighted DiD models yield nearly identical estimates.

---

### Causal Insight

The remaining gap between DiD (~ -4.6) and the true ATT (-6.538) is therefore unlikely to be driven by observable baseline differences.

Instead, the residual bias likely stems from:

- Time-varying differences between groups
- Imperfect parallel trends
- Structural targeting interacting with seasonal demand patterns

---

### Business Interpretation

Even after statistically adjusting for customer mix, the estimated incremental margin impact remains approximately **-$4.60 per exposed user-day**.

This suggests the observed effect is not driven by simple targeting imbalance but reflects a structural change in profitability during the promotion window.

---

### Key Takeaway

- DiD partially corrects selection bias.
- Fixed effects do not materially change the estimate.
- IPW fails due to lack of overlap.
- Weighted DiD confirms robustness of the unweighted DiD estimate.

This reinforces that identification assumptions — particularly overlap and parallel trends — are central to causal credibility.